In [39]:
import numpy as np
from scipy.signal import butter, sosfiltfilt, welch
from scipy.stats import kurtosis


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py 


## Load Test file (Mini, for exploration)

In [40]:
path='/home/eivind/data-science-bootcamp/project-seismic/Data/Mini/miniPNW_metadata.csv'
df=pd.read_csv(path)

pathwf='/home/eivind/data-science-bootcamp/project-seismic/Data/Mini/miniPNW_waveforms.hdf5'
wf=h5py.File(pathwf,'r')


## Data Cleaning


In [41]:
#Define which columns to keep. 
KEEP_ID    = ["trace_name", "event_id", "station_code", "source_type"]
KEEP_WAVE  = ["trace_sampling_rate_hz", "trace_P_arrival_sample", "trace_S_arrival_sample"]
KEEP_LEAK  = ["source_depth_km", "source_latitude_deg", "source_longitude_deg",
              "source_origin_time", "preferred_source_magnitude", "preferred_source_magnitude_type"]
#Keep only earthquakes and explosions
#Delete dead signals that have no P arrival
def clean(df):
    df = df[df.source_type.isin(["earthquake", "explosion"])]
    df = df.dropna(subset=["trace_P_arrival_sample"])
    return df[KEEP_ID + KEEP_WAVE + KEEP_LEAK].reset_index(drop=True)

In [42]:
df=clean(df)

## Start finding features


### Processing


In [43]:

#Each waveform has three channels. E is the east-west direction, N is north-south, Z is vertical. Noe that we only ever care about N^2+E^2, so it makes no difference if these are rotated wrt to the labels.
CH = {"E": 0, "N": 1, "Z": 2}    

#Define a basic bandpass filter
def bandpass(x, fs, lo, hi, order=4): 
    sos = butter(order, [lo, hi], btype="band", fs=fs, output="sos")
    return sosfiltfilt(sos, x, axis=-1)

#preprocessing function that substracts the mean and tapers the signal
def preprocess(tr3, fs):
    x = tr3 - tr3.mean(axis=-1, keepdims=True)
    n = x.shape[-1]
    taper = np.ones(n); k = int(0.05 * n)
    taper[:k] = np.hanning(2*k)[:k]; taper[-k:] = np.hanning(2*k)[k:]
    return x * taper

#computes the RMS of the waveform in a given window 
def window_rms(x, start, length): 
    start = max(int(start), 0)
    seg = x[..., start:start + int(length)]
    return np.sqrt(np.mean(seg**2, axis=-1)) if seg.shape[-1] > 0 else np.nan


### First and Second feautre: Amplitude features
 log(P_z/S_h) and log(horizontal/vertical) at the S-wave


In [44]:
def amp_features(tr3, fs, p_samp, s_samp, lo=4, hi=8, p_len_s=1.5, s_len_s=3.0, noise_len_s=3.0, guard_s=0.5):
    tr3 = tr3.astype(float)          # Turn entries into floats
    n_dead = int((np.abs(tr3).max(axis=1) == 0).sum())   # count number of deads by measuring absolute value
    out = dict(log_ps=np.nan, log_hv_s=np.nan, snr_p=np.nan, snr_s=np.nan, n_dead=n_dead) #return nan if dead channels
    if np.isnan(s_samp) or n_dead > 0: #if no s or dead channels, return nan
        return out

    x = bandpass(preprocess(tr3, fs), fs, lo, hi) #filter signal
    p, s0 = int(p_samp), int(s_samp) #make arrivel time indices integers
    p_len = min(int(p_len_s * fs), s0 - p) # define length of p window as standard or shorter if S is too close. 
    if p_len < int(0.3 * fs):  #If P and S too close, Kill
        return out

    z = x[CH["Z"]]  #vertical channel.
    h = np.sqrt(x[CH["N"]]**2 + x[CH["E"]]**2)  #horizontal channel. 
    s_len, guard = int(s_len_s * fs), int(guard_s * fs)  #turn seconds into indices

    noise = window_rms(z, p - guard - int(noise_len_s * fs), int(noise_len_s * fs))   #noise measure
    P   = window_rms(z, p, p_len) # P z rms
    S_h = window_rms(h, s0, s_len) # S h rms
    S_z = window_rms(z, s0, s_len) # S z rms

    #return amplitude ratios 
    out.update(
        log_ps=np.log10(P / S_h),
        log_hv_s=np.log10(S_h / S_z),
        snr_p=np.log10(P / noise),
        snr_s=np.log10(S_h / noise),
    )
    return out

### Third and fourth feature: envelope features
 
rise_time and kurtosis


In [45]:
def envelope_features(tr3, fs, p_samp, s_samp, lo=2, hi=10,pre_s=1.0, post_s=10.0, smooth_s=0.2):
    
    tr3 = tr3.astype(float)                           #change entries to float
    
    out = dict(rise_time=np.nan, env_kurtosis=np.nan) # the return if no S wave or dead Z channel
    if np.isnan(s_samp):                              #default if no S wave registered
        return out
    if np.abs(tr3[CH["Z"]]).max() == 0:             #return default if not Z channel
        return out

    x = bandpass(preprocess(tr3, fs), fs, lo, hi)  #filter all channels
    env = np.sqrt((x**2).sum(axis=0))       #envelope is old name, we are really just taking the 3d amplitude
    k = max(int(smooth_s * fs), 1)          #Take the max of the smooth_s (in ticks) variable and 1. 
    env = np.convolve(env, np.ones(k) / k, mode="same") #smooth envelope with a convolution

    a = max(int(p_samp - pre_s * fs), 0)  #a is the point at which we start the analysis. p arrival minus some value pre_s
    b = min(int(s_samp + post_s * fs), len(env)) #b is the point at which end the analysis
    e = env[a:b]  #slice the envelop
    if len(e) < fs:  #default if we don't have at leas 1 second
        return out

    # rise time: 10% -> 90% of max, on the way up to the peak
    i_peak = e.argmax() #find the highest peak (usually P)
    rise = e[:i_peak + 1] #slice into the portion before
    i10 = np.argmax(rise >= 0.1 * e.max()) #returns the first point at which e>= 0.1 of peak
    i90 = np.argmax(rise >= 0.9 * e.max()) #returns the first point at which e>= 0.9 of peak
    out["rise_time"] = (i90 - i10) / fs #record the time between these two points

    # excess kurtosis of the envelope over the event window
    out["env_kurtosis"] = kurtosis(e) #record the kurtosis of the envelop. 
    return out

### Fifth feature: spectral energy ratio (low fq/high fq)

In [46]:
def spectral_features(tr3, fs, p_samp, pre_s=1.0, win_s=20.0,low=(1, 5), high=(5, 15)):  #figure out if these frequency intervals are reasonabe

    tr3 = tr3.astype(float) 
    
    out = dict(log_lowhigh=np.nan)          #Set default ouput to nan
    dead = np.abs(tr3).max(axis=1) == 0     #check for dead channels
    if dead.any():                          # return nan if channels are dead     
        return out
    

    x = preprocess(tr3, fs)       #preprocess the signal. #Important to not smooth it to not delete high frequencies etc.
    a = max(int(p_samp - pre_s * fs), 0) # beginning of the window
    b = min(int(p_samp + win_s * fs), x.shape[1]) # end of the window
    seg = x[:, a:b]                               # slice each channel 
    if seg.shape[1] < 2 * fs:                      #return default if window is shorter than 2sec
        return out

    f, pxx = welch(seg, fs=fs, nperseg=int(2 * fs), axis=-1) #returns the fourier tranfsorm f (with Welch method) and pxx is the power (squared frequencies)
    pxx = pxx.sum(axis=0)                                    #sum square powers

    m_low  = (f >= low[0])  & (f < low[1]) #true/false list or array of where the frequencies are in the low range
    m_high = (f >= high[0]) & (f < high[1]) #ditto for high range
    e_low  = np.trapezoid(pxx[m_low],  f[m_low]) # integrate power with trapezoid rule in the low frequency range
    e_high = np.trapezoid(pxx[m_high], f[m_high]) #integrate power in high frequency range. 
    out["log_lowhigh"] = np.log10(e_low / e_high) #we record the log(low/high) ratio
    return out

### Sixth feature:  Difference in arrival time (this one is sketchy - potentially leaky!!!)


In [47]:
df["sp_time"] = (df.trace_S_arrival_sample - df.trace_P_arrival_sample) / df.trace_sampling_rate_hz

In [48]:
df

,trace_name,event_id,station_code,source_type,trace_sampling_rate_hz,trace_P_arrival_sample,trace_S_arrival_sample,source_depth_km,source_latitude_deg,source_longitude_deg,source_origin_time,preferred_source_magnitude,preferred_source_magnitude_type,sp_time
0,"bucket1$0,:3,:15001",uw61669232,MOX,earthquake,100,5748.0,6314.0,23.300,46.560,-119.797,2020-09-07T03:44:14.690000Z,1.30,ml,5.66
1,"bucket1$1,:3,:15001",uw60888282,G05D,earthquake,100,5697.0,6217.0,-0.947,45.371,-121.708,2014-10-08T15:39:31.330000Z,1.67,ml,5.20
2,"bucket1$2,:3,:15001",uw61361706,CCRK,earthquake,100,6216.0,7105.0,13.520,46.165,-120.543,2017-12-30T04:37:46.870000Z,2.46,ml,8.89
3,"bucket1$3,:3,:15001",uw61639436,WA2,earthquake,100,5524.0,5901.0,16.370,46.542,-119.589,2020-06-09T23:37:10.420000Z,1.59,ml,3.77
4,"bucket1$4,:3,:15001",uw61735446,SIFT,earthquake,100,5225.0,5387.0,12.380,46.857,-121.941,2021-05-24T10:42:37.810000Z,0.83,ml,1.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,"bucket2$495,:3,:15001",uw10831163,F04D,explosion,100,5251.0,5526.0,-0.385,46.206,-123.027,2011-12-10T00:10:12.140000Z,1.77,ml,2.75
996,"bucket2$496,:3,:15001",uw61553191,DAVN,explosion,100,6057.0,6781.0,-0.430,47.363,-117.809,2019-10-02T22:47:53.660000Z,2.14,ml,7.24
997,"bucket2$497,:3,:15001",uw61357892,OMAK,explosion,100,7298.0,9210.0,-0.370,49.364,-120.493,2018-03-27T21:16:10.820000Z,2.07,ml,19.12
998,"bucket2$498,:3,:15001",uw61876121,SHUK,explosion,100,5677.0,6203.0,-1.100,49.134,-121.975,2022-08-29T20:16:57.870000Z,0.84,md,5.26


### Add remaining features to the data frame

In [ ]:
# Given a trace name, load the wavefunction
def load_trace(wf, trace_name): 
    bucket, sl = trace_name.split("$") #Splits the trace name into bucket name and event number.
    return wf[f"data/{bucket}"][int(sl.split(",")[0])] #returns the waveform. First bracket selects the bucket, second the event by number.

In [ ]:
#Build new dataframe containing new features
def build_features(df, wf):
    rows = []
    for _, r in df.iterrows():                                                                          #iterate over rows
        tr3 = load_trace(wf, r.trace_name)                                                              #load waveform
        fs, p, s = r.trace_sampling_rate_hz, r.trace_P_arrival_sample, r.trace_S_arrival_sample         #Get basic specs, sampling frequency and arrival times
        rows.append({**amp_features(tr3, fs, p, s),                                                     #Unpack the dictionary returned by the feature functions. Append to Rows
                     **envelope_features(tr3, fs, p, s),
                     **spectral_features(tr3, fs, p)})
    F = pd.concat([df, pd.DataFrame(rows)], axis=1)                                                    #concatenate these new features with the original data frame
    F["sp_time"] = (F.trace_S_arrival_sample - F.trace_P_arrival_sample) / F.trace_sampling_rate_hz    # Add the 6th (potentially sketchy/leaky feature!), the distance between the arrival times. I don't think we want to train on this, since it might be a proxy for distance, which learnes mine locations, which might not generalize. 
    return F



In [52]:
F = build_features(clean(df), wf)

In [53]:
F

,trace_name,event_id,station_code,source_type,trace_sampling_rate_hz,trace_P_arrival_sample,trace_S_arrival_sample,source_depth_km,source_latitude_deg,source_longitude_deg,...,preferred_source_magnitude_type,log_ps,log_hv_s,snr_p,snr_s,n_dead,rise_time,env_kurtosis,log_lowhigh,sp_time
0,"bucket1$0,:3,:15001",uw61669232,MOX,earthquake,100,5748.0,6314.0,23.300,46.560,-119.797,...,ml,0.113556,0.208802,1.379931,1.266375,0,0.19,7.068796,-0.369913,5.66
1,"bucket1$1,:3,:15001",uw60888282,G05D,earthquake,100,5697.0,6217.0,-0.947,45.371,-121.708,...,ml,-0.600555,0.312756,1.359164,1.959720,0,5.23,5.424048,0.331651,5.20
2,"bucket1$2,:3,:15001",uw61361706,CCRK,earthquake,100,6216.0,7105.0,13.520,46.165,-120.543,...,ml,-0.707312,0.193750,1.299132,2.006444,0,6.22,1.170810,1.221780,8.89
3,"bucket1$3,:3,:15001",uw61639436,WA2,earthquake,100,5524.0,5901.0,16.370,46.542,-119.589,...,ml,-1.025464,0.417800,1.007305,2.032769,0,1.59,2.274190,0.181536,3.77
4,"bucket1$4,:3,:15001",uw61735446,SIFT,earthquake,100,5225.0,5387.0,12.380,46.857,-121.941,...,ml,-0.670146,0.879596,1.465294,2.135439,0,1.74,24.965538,-0.582629,1.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,"bucket2$495,:3,:15001",uw10831163,F04D,explosion,100,5251.0,5526.0,-0.385,46.206,-123.027,...,ml,-0.536586,-0.193478,0.219991,0.756577,0,2.33,-1.035300,2.832400,2.75
996,"bucket2$496,:3,:15001",uw61553191,DAVN,explosion,100,6057.0,6781.0,-0.430,47.363,-117.809,...,ml,-0.881321,0.300171,0.598367,1.479689,0,10.28,-0.851099,0.889395,7.24
997,"bucket2$497,:3,:15001",uw61357892,OMAK,explosion,100,7298.0,9210.0,-0.370,49.364,-120.493,...,ml,-0.689160,0.068046,0.240998,0.930157,0,20.64,3.373522,-0.150908,19.12
998,"bucket2$498,:3,:15001",uw61876121,SHUK,explosion,100,5677.0,6203.0,-1.100,49.134,-121.975,...,md,-0.339411,0.286994,0.528876,0.868288,0,1.40,-0.262027,0.216049,5.26


In [55]:
F["trace_S_arrival_sample"].isna()

0      False
1      False
2      False
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Name: trace_S_arrival_sample, Length: 1000, dtype: bool